In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

In [2]:
df = pd.read_csv("LengthOfStay.csv",index_col=0)
df.head()

,vdate,rcount,gender,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,...,glucose,bloodureanitro,creatinine,bmi,pulse,respiration,secondarydiagnosisnonicd9,discharged,facid,lengthofstay
eid,,,,,,,,,,,,,,,,,,,,,
1,8/29/2012,0,F,0,0,0,0,0,0,0,...,192.476918,12.0,1.390722,30.432418,96,6.5,4,9/1/2012,B,3
2,5/26/2012,5+,F,0,0,0,0,0,0,0,...,94.078507,8.0,0.943164,28.460516,61,6.5,1,6/2/2012,A,7
3,9/22/2012,1,F,0,0,0,0,0,0,0,...,130.530524,12.0,1.065750,28.843812,64,6.5,2,9/25/2012,B,3
4,8/9/2012,0,F,0,0,0,0,0,0,0,...,163.377028,12.0,0.906862,27.959007,76,6.5,1,8/10/2012,A,1
5,12/20/2012,0,F,0,0,0,1,0,1,0,...,94.886654,11.5,1.242854,30.258927,67,5.6,2,12/24/2012,E,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 1 to 100000
Data columns (total 27 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   vdate                       100000 non-null  object 
 1   rcount                      100000 non-null  object 
 2   gender                      100000 non-null  object 
 3   dialysisrenalendstage       100000 non-null  int64  
 4   asthma                      100000 non-null  int64  
 5   irondef                     100000 non-null  int64  
 6   pneum                       100000 non-null  int64  
 7   substancedependence         100000 non-null  int64  
 8   psychologicaldisordermajor  100000 non-null  int64  
 9   depress                     100000 non-null  int64  
 10  psychother                  100000 non-null  int64  
 11  fibrosisandother            100000 non-null  int64  
 12  malnutrition                100000 non-null  int64  
 13  hemo               

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.drop(columns=['vdate','discharged','facid','secondarydiagnosisnonicd9'], inplace=True)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
x = df.drop(['lengthofstay'],axis=1)
y = df['lengthofstay']

In [9]:
num_cols = x.select_dtypes(include= 'number').columns
num_cols

Index(['dialysisrenalendstage', 'asthma', 'irondef', 'pneum',
       'substancedependence', 'psychologicaldisordermajor', 'depress',
       'psychother', 'fibrosisandother', 'malnutrition', 'hemo', 'hematocrit',
       'neutrophils', 'sodium', 'glucose', 'bloodureanitro', 'creatinine',
       'bmi', 'pulse', 'respiration'],
      dtype='object')

In [10]:
cat_cols = x.select_dtypes(include= 'object').columns
cat_cols

Index(['rcount', 'gender'], dtype='object')

In [11]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

scaler = MinMaxScaler()

num_pipline = Pipeline(steps=[('Scaler',scaler)])
num_pipline

,steps,"[('Scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,feature_range,"(0, ...)"
,copy,True
,clip,False


In [12]:
from category_encoders import BinaryEncoder

BE = BinaryEncoder()

Cat_Pipeline = Pipeline(steps= [ ('Binary Encoder', BE) ])
Cat_Pipeline

,steps,"[('Binary Encoder', ...)]"
,transform_input,None
,memory,None
,verbose,False
,verbose,0
,cols,None
,mapping,None
,drop_invariant,False
,return_df,True
,base,2
,handle_unknown,'value'


In [13]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(transformers= [ ('Num Pipeline', num_pipline, num_cols),
                                                  ('Cat Pipeline', Cat_Pipeline, cat_cols) ],
                                                  remainder= 'passthrough')
preprocessing

,transformers,"[('Num Pipeline', ...), ('Cat Pipeline', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,feature_range,"(0, ...)"
,copy,True
,clip,False


In [14]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_validate

models = [ ('Linear Regression', LinearRegression()),
           ('Lasso', Lasso(random_state = 42)),
           ('Ridge', Ridge(random_state= 42)),
           ('ElasticNet', ElasticNet(random_state= 42)),
           ('Polynomial', PolynomialFeatures(degree = 2)),
           ('Knn Regressor', KNeighborsRegressor(n_neighbors= 5)),
           #('SVM', SVR(kernel= 'linear')),
           ('DT', DecisionTreeRegressor(random_state = 42)),
           ('RF', RandomForestRegressor(random_state = 42, n_jobs= -1)),
           ('XGB', XGBRegressor(random_state = 42, n_jobs= -1)),
           ('CB', CatBoostRegressor(random_state = 42)),
           ('LGBM', LGBMRegressor(random_state = 42, n_jobs= -1)),
           ]

for model in models:

    model_pipeline = Pipeline(steps= [ ('Preprocessing', preprocessing), ('Model', model[1])])

    if model[0] == 'Polynomial':
        model_pipeline = Pipeline(steps= [ ('Preprocessing', preprocessing), ('Polynomial', model[1]), ('Model', Ridge(random_state= 42))])

    cv_results = cross_validate(estimator= model_pipeline, X= x, y= y, cv= 5, scoring= 'r2', return_train_score= True)

    print(model[0])
    print('Train R2 Score :', round(cv_results['train_score'].mean() * 100, 3))
    print('Test R2 Score :', round(cv_results['test_score'].mean() * 100, 3))
    print('Average Training Time :', round(cv_results['fit_time'].mean(), 4))
    print('-' * 50)

Linear Regression
Train R2 Score : 53.63
Test R2 Score : 53.621
Average Training Time : 0.308
--------------------------------------------------
Lasso
Train R2 Score : 0.0
Test R2 Score : -0.001
Average Training Time : 0.2405
--------------------------------------------------
Ridge
Train R2 Score : 53.63
Test R2 Score : 53.621
Average Training Time : 0.2522
--------------------------------------------------
ElasticNet
Train R2 Score : 1.3
Test R2 Score : 1.298
Average Training Time : 0.3655
--------------------------------------------------
Polynomial
Train R2 Score : 86.662
Test R2 Score : 86.456
Average Training Time : 0.7739
--------------------------------------------------
Knn Regressor
Train R2 Score : 89.0
Test R2 Score : 82.877
Average Training Time : 0.1529
--------------------------------------------------
DT
Train R2 Score : 100.0
Test R2 Score : 85.048
Average Training Time : 1.3691
--------------------------------------------------
RF
Train R2 Score : 98.952
Test R2 Score 

c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008755 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1912
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 25
[LightGBM] [Info] Start training from score 4.002987


c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002884 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1919
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 25
[LightGBM] [Info] Start training from score 3.998838


c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008390 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1919
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 25
[LightGBM] [Info] Start training from score 3.999975


c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002722 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1971
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 25
[LightGBM] [Info] Start training from score 4.001462
LGBM
Train R2 Score : 96.419
Test R2 Score : 96.057
Average Training Time : 0.4199
--------------------------------------------------


c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\MyAnaconda\envs\oop\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [15]:
catboost_pipeline = Pipeline(steps= [ ('Preprocessing', preprocessing), ('Model', CatBoostRegressor(random_state = 42))])
catboost_pipeline

,steps,"[('Preprocessing', ...), ('Model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Num Pipeline', ...), ('Cat Pipeline', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'Model__depth': [4, 6, 8],
    'Model__learning_rate': [0.01, 0.03, 0.05],
    'Model__iterations': [500, 800, 1200]
    }

# Turn off parallel processing entirely
random_search = RandomizedSearchCV(
    catboost_pipeline, 
    param_distributions=param_grid, 
    cv=5, 
    scoring='r2', 
    random_state=42, 
    return_train_score=True, 
    n_jobs=1  # Sequential processing
)

random_search.fit(x, y)

0:	learn: 2.2869234	total: 15.1ms	remaining: 18.1s
1:	learn: 2.2159264	total: 19.6ms	remaining: 11.7s
2:	learn: 2.1498330	total: 24ms	remaining: 9.59s
3:	learn: 2.0883826	total: 27.8ms	remaining: 8.31s
4:	learn: 2.0313224	total: 31.4ms	remaining: 7.5s
5:	learn: 1.9784080	total: 34.4ms	remaining: 6.84s
6:	learn: 1.9294022	total: 38.2ms	remaining: 6.51s
7:	learn: 1.8837511	total: 41.8ms	remaining: 6.23s
8:	learn: 1.8410537	total: 46ms	remaining: 6.08s
9:	learn: 1.8022495	total: 49.3ms	remaining: 5.86s
10:	learn: 1.7654120	total: 53.3ms	remaining: 5.76s
11:	learn: 1.7309921	total: 56.9ms	remaining: 5.63s
12:	learn: 1.6997182	total: 60.1ms	remaining: 5.48s
13:	learn: 1.6690778	total: 63.2ms	remaining: 5.35s
14:	learn: 1.6404230	total: 67ms	remaining: 5.29s
15:	learn: 1.6146763	total: 70ms	remaining: 5.18s
16:	learn: 1.5909125	total: 74ms	remaining: 5.15s
17:	learn: 1.5672564	total: 77.1ms	remaining: 5.06s
18:	learn: 1.5456073	total: 80.5ms	remaining: 5s
19:	learn: 1.5253749	total: 83.8ms	r

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'Model__depth': [4, 6, ...], 'Model__iterations': [500, 800, ...], 'Model__learning_rate': [0.01, 0.03, ...]}"
,n_iter,10
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [18]:
random_search.best_score_

np.float64(0.9720784995304659)

In [19]:
random_search.best_params_

{'Model__learning_rate': 0.05, 'Model__iterations': 1200, 'Model__depth': 6}

In [20]:
train_scores = random_search.cv_results_['mean_train_score'] * 100
train_scores

array([96.71631865, 97.07973295, 91.11498626, 96.04206204, 86.03541174,
       97.11589069, 97.34115353, 97.55284304, 94.48058656, 96.99725869])

In [21]:
test_scores = random_search.cv_results_['mean_test_score'] * 100
test_scores

array([96.55527593, 96.9182431 , 91.00767584, 95.76398506, 85.98598835,
       96.9485944 , 97.11784415, 97.20784995, 94.372082  , 96.71545309])

In [22]:
px.scatter(x= train_scores, y= test_scores, labels= {'x' : 'Train Scores', 'y' : 'Test Scores'})

In [23]:
catboost_final_pipeline =  Pipeline(steps= [ ('Preprocessing', preprocessing), ('Model', CatBoostRegressor(random_state = 42))])
catboost_final_pipeline

,steps,"[('Preprocessing', ...), ('Model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Num Pipeline', ...), ('Cat Pipeline', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [24]:
catboost_final_pipeline.fit(x, y)

Learning rate set to 0.084758
0:	learn: 2.2180004	total: 7.4ms	remaining: 7.39s
1:	learn: 2.0904256	total: 14.1ms	remaining: 7.04s
2:	learn: 1.9765605	total: 20.5ms	remaining: 6.8s
3:	learn: 1.8770745	total: 26.3ms	remaining: 6.54s
4:	learn: 1.7860127	total: 31.9ms	remaining: 6.35s
5:	learn: 1.7041964	total: 37.2ms	remaining: 6.17s
6:	learn: 1.6325004	total: 43.3ms	remaining: 6.14s
7:	learn: 1.5669052	total: 48.1ms	remaining: 5.96s
8:	learn: 1.5086945	total: 52.4ms	remaining: 5.77s
9:	learn: 1.4575266	total: 58.1ms	remaining: 5.75s
10:	learn: 1.4121503	total: 64.3ms	remaining: 5.78s
11:	learn: 1.3723683	total: 69.5ms	remaining: 5.72s
12:	learn: 1.3351111	total: 77.4ms	remaining: 5.87s
13:	learn: 1.3027495	total: 83.1ms	remaining: 5.85s
14:	learn: 1.2719277	total: 89.6ms	remaining: 5.88s
15:	learn: 1.2434238	total: 94.5ms	remaining: 5.81s
16:	learn: 1.2175118	total: 99.2ms	remaining: 5.74s
17:	learn: 1.1928177	total: 104ms	remaining: 5.69s
18:	learn: 1.1714107	total: 110ms	remaining: 5.

,steps,"[('Preprocessing', ...), ('Model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Num Pipeline', ...), ('Cat Pipeline', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [31]:
x.sample(random_state= 41)

,rcount,gender,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,psychother,...,hemo,hematocrit,neutrophils,sodium,glucose,bloodureanitro,creatinine,bmi,pulse,respiration
eid,,,,,,,,,,,,,,,,,,,,,
48284,0,M,0,0,0,0,0,0,0,0,...,0,14.9,7.8,135.09393,130.337141,14.0,0.949112,29.498411,67,6.5


In [33]:
catboost_final_pipeline.predict(x.sample(random_state= 41)).round(0)[0]

np.float64(1.0)

In [27]:
import joblib

joblib.dump(catboost_final_pipeline, 'Catboost_Model_LOS.pkl')

['Catboost_Model_LOS.pkl']

In [28]:
df.to_csv('Cleaned_df.csv', index=False)

In [29]:
import streamlit as st

In [30]:
! streamlit run app.py

^C


In [34]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

# ── Page config MUST be the first Streamlit command ────────────────────────────
st.set_page_config(
    page_title="LOS Predictor - Hospital Length of Stay",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS for better styling ──────────────────────────────────────────────
st.markdown("""
<style>
    /* Main container styling */
    .main-header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.5rem;
        border-radius: 10px;
        margin-bottom: 2rem;
        color: white;
        text-align: center;
    }
    
    .main-header h1 {
        margin: 0;
        font-size: 2.5rem;
        font-weight: 700;
    }
    
    .main-header p {
        margin: 0.5rem 0 0 0;
        opacity: 0.9;
    }
    
    /* Card styling */
    .prediction-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        border-radius: 20px;
        padding: 2rem;
        color: white;
        text-align: center;
        box-shadow: 0 10px 30px rgba(0,0,0,0.2);
        margin: 1rem 0;
    }
    
    .prediction-number {
        font-size: 5rem;
        font-weight: 800;
        margin: 1rem 0;
        line-height: 1;
    }
    
    .prediction-label {
        font-size: 1rem;
        text-transform: uppercase;
        letter-spacing: 2px;
        opacity: 0.9;
    }
    
    .stay-category {
        font-size: 1.2rem;
        font-weight: 600;
        margin-top: 1rem;
        padding: 0.5rem;
        border-radius: 10px;
        background: rgba(255,255,255,0.2);
    }
    
    /* Section headers */
    .section-header {
        font-size: 1.2rem;
        font-weight: 600;
        color: #667eea;
        margin: 1rem 0 1rem 0;
        padding-bottom: 0.5rem;
        border-bottom: 2px solid #667eea;
    }
    
    /* Info boxes */
    .info-box {
        background: #f8f9fa;
        border-left: 4px solid #667eea;
        padding: 1rem;
        border-radius: 8px;
        margin: 1rem 0;
    }
    
    /* Risk factor chips */
    .risk-chip {
        display: inline-block;
        background: #ff6b6b;
        color: white;
        padding: 0.3rem 0.8rem;
        border-radius: 20px;
        margin: 0.2rem;
        font-size: 0.85rem;
        font-weight: 500;
    }
    
    /* Button styling */
    .stButton > button {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        border: none;
        padding: 0.75rem 1.5rem;
        font-weight: 600;
        width: 100%;
        transition: all 0.3s ease;
    }
    
    .stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 5px 15px rgba(102,126,234,0.4);
    }
</style>
""", unsafe_allow_html=True)

# ── Model loader with column extraction ────────────────────────────────────────
@st.cache_resource
def load_model_and_features():
    """Load the CatBoost model and extract expected features"""
    try:
        from catboost import CatBoostRegressor
        model = CatBoostRegressor()
        model.load_model('Catboost_Model_LOS.pkl')
        
        # Try to get expected feature names
        expected_features = None
        try:
            if hasattr(model, 'feature_names_'):
                expected_features = model.feature_names_
            elif hasattr(model, 'get_feature_names'):
                expected_features = model.get_feature_names()
        except:
            pass
            
        return model, expected_features
    except:
        try:
            import joblib
            model = joblib.load('Catboost_Model_LOS.pkl')
            return model, None
        except:
            try:
                with open('Catboost_Model_LOS.pkl', 'rb') as f:
                    model = pickle.load(f)
                return model, None
            except:
                st.error("❌ Failed to load model. Please check if 'Catboost_Model_LOS.pkl' exists.")
                return None, None

# ── Helper functions ───────────────────────────────────────────────────────────
def predict_los(model, input_data, expected_features=None):
    """Make prediction and return with interpretation"""
    try:
        # Ensure all expected columns are present
        if expected_features:
            # Check which expected features are missing
            missing_cols = set(expected_features) - set(input_data.columns)
            if missing_cols:
                st.warning(f"Adding missing columns with default values: {missing_cols}")
                for col in missing_cols:
                    input_data[col] = 0
            
            # Reorder columns to match model expectations
            input_data = input_data[expected_features]
        
        prediction = model.predict(input_data)[0]
        # Round to nearest whole number for days
        los = max(0, round(float(prediction)))  # Removed decimal, now whole number
        
        # Categorize LOS
        if los <= 3:
            category = "Short Stay"
            description = "Routine admission, expected quick recovery"
            icon = "🟢"
        elif los <= 7:
            category = "Moderate Stay"
            description = "Standard inpatient care duration"
            icon = "🟡"
        elif los <= 14:
            category = "Extended Stay"
            description = "Complex condition requiring longer monitoring"
            icon = "🟠"
        else:
            category = "Prolonged Stay"
            description = "Severe condition requiring intensive care"
            icon = "🔴"
        
        return los, category, description, icon
    except Exception as e:
        st.error(f"Prediction error: {str(e)}")
        st.info("💡 Tip: Check if all required fields are filled correctly")
        return None, None, None, None

def get_risk_factors(data):
    """Extract and return active risk factors"""
    risk_mapping = {
        'dialysisrenalendstage': 'Dialysis/Renal Failure',
        'asthma': 'Asthma',
        'irondef': 'Iron Deficiency',
        'pneum': 'Pneumonia',
        'substancedependence': 'Substance Dependence',
        'psychologicaldisordermajor': 'Major Psychiatric Disorder',
        'depress': 'Depression',
        'psychother': 'Other Psychiatric Disorder',
        'fibrosisandother': 'Fibrosis/Pulmonary Disease',
        'malnutrition': 'Malnutrition'
    }
    
    active_risks = []
    for key, label in risk_mapping.items():
        if key in data and data[key] == 1:
            active_risks.append(label)
    
    return active_risks

# ── Main app ───────────────────────────────────────────────────────────────────
def main():
    # Header
    st.markdown("""
    <div class="main-header">
        <h1>🏥 Hospital Length of Stay Predictor</h1>
        <p>AI-powered prediction using CatBoost regression model</p>
    </div>
    """, unsafe_allow_html=True)
    
    # Load model
    with st.spinner("Loading AI model..."):
        model, expected_features = load_model_and_features()
    
    if model is None:
        st.stop()
    
    # Display expected features if available (for debugging)
    if expected_features and st.sidebar.checkbox("Show Model Info", value=False):
        with st.sidebar.expander("📊 Model Features"):
            st.write(f"Expected {len(expected_features)} features:")
            st.write(expected_features)
    
    # ── SIDEBAR ────────────────────────────────────────────────────────────────
    with st.sidebar:
        st.markdown("## 🏥 Patient Information")
        st.markdown("---")
        
        # Demographics Section
        st.markdown("### 👤 Demographics")
        gender = st.selectbox("Gender", ["F", "M"], help="Patient's biological sex")
        
        rcount = st.selectbox(
            "Readmission Count (last 180 days)",
            ["0", "1", "2", "3", "4", "5+"],
            help="Number of prior hospital readmissions"
        )
        
        # BMI Input
        st.markdown("### 📏 Body Metrics")
        col1, col2 = st.columns(2)
        with col1:
            height_ft = st.number_input("Height (feet)", min_value=0.0, max_value=None, value=5.5, step=0.1, help="Height in feet")
        with col2:
            height_in = st.number_input("Height (inches)", min_value=0, max_value=None, value=0, step=1, help="Additional inches")
        
        weight_lbs = st.number_input("Weight (lbs)", min_value=0.0, max_value=None, value=150.0, step=1.0, help="Weight in pounds")
        
        # Calculate BMI
        height_total_inches = (height_ft * 12) + height_in
        if height_total_inches > 0:
            bmi = (weight_lbs / (height_total_inches ** 2)) * 703
            bmi = round(bmi, 1)
        else:
            bmi = 0.0
        
        # BMI Status (only show if valid)
        if bmi > 0:
            if bmi < 18.5:
                bmi_status = "⚠️ Underweight"
            elif bmi < 25:
                bmi_status = "✅ Normal"
            elif bmi < 30:
                bmi_status = "⚠️ Overweight"
            else:
                bmi_status = "⚠️ Obese"
            st.info(f"📊 BMI: **{bmi}** ({bmi_status})")
        else:
            st.info(f"📊 BMI: **{bmi}** (Invalid height/weight)")
        
        st.markdown("---")
        
        # Lab Results Section - NO RESTRICTIONS
        st.markdown("### 🧪 Lab Results")
        col1, col2 = st.columns(2)
        with col1:
            hematocrit = st.number_input("Hematocrit (%)", value=36.0, step=0.5, help="Red blood cell volume percentage")
            sodium = st.number_input("Sodium (mEq/L)", value=138.0, step=0.5, help="Serum sodium level")
            glucose = st.number_input("Glucose (mg/dL)", value=110.0, step=1.0, help="Blood glucose level")
            creatinine = st.number_input("Creatinine (mg/dL)", value=1.0, step=0.05, help="Kidney function marker")
        
        with col2:
            neutrophils = st.number_input("Neutrophils (×10³/µL)", value=6.0, step=0.1, help="White blood cell subtype")
            bloodureanitro = st.number_input("BUN (mg/dL)", value=18.0, step=0.5, help="Blood urea nitrogen level")
            hemo = st.number_input("Hemoglobin (g/dL)", value=12.5, step=0.1, help="Oxygen-carrying protein")
        
        st.markdown("---")
        
        # Vital Signs - NO RESTRICTIONS
        st.markdown("### 💓 Vital Signs")
        col1, col2 = st.columns(2)
        with col1:
            pulse = st.number_input("Heart Rate (bpm)", value=80, step=1, help="Beats per minute")
        with col2:
            respiration = st.number_input("Respiration Rate", value=16.0, step=0.5, help="Breaths per minute")
        
        st.markdown("---")
        
        # Comorbidities Section
        st.markdown("### 🩺 Comorbidities")
        st.caption("Select 1 if present, 0 if absent")
        
        col1, col2 = st.columns(2)
        with col1:
            dialysis = st.selectbox("Dialysis/Renal", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            asthma = st.selectbox("Asthma", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            irondef = st.selectbox("Iron Deficiency", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            pneum = st.selectbox("Pneumonia", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            substance = st.selectbox("Substance Dependence", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
        
        with col2:
            psychmajor = st.selectbox("Major Psychiatric", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            depress = st.selectbox("Depression", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            psychother = st.selectbox("Other Psychiatric", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            fibrosis = st.selectbox("Fibrosis/Pulmonary", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
            malnutrition = st.selectbox("Malnutrition", [0, 1], format_func=lambda x: "✓ Present" if x else "✗ Absent")
        
        st.markdown("---")
        
        # Predict Button
        predict_button = st.button("🔮 PREDICT LENGTH OF STAY", type="primary", use_container_width=True)
    
    # ── MAIN CONTENT AREA ──────────────────────────────────────────────────────
    # Create two columns for main content
    col1, col2 = st.columns([1, 1])
    
    with col1:
        st.markdown('<div class="section-header">📋 Patient Summary</div>', unsafe_allow_html=True)
        
        # Create summary dataframe
        rcount_display = rcount
        
        # Format BMI display
        bmi_display = f"{bmi}" if bmi > 0 else "Invalid"
        if bmi > 0:
            bmi_display = f"{bmi} ({bmi_status})"
        
        summary_data = {
            "Category": ["Demographics", "Demographics", "Demographics",
                        "Labs", "Labs", "Labs", "Labs", "Labs", "Labs", "Labs",
                        "Vitals", "Vitals",
                        "Comorbidities", "Comorbidities", "Comorbidities", 
                        "Comorbidities", "Comorbidities", "Comorbidities", 
                        "Comorbidities", "Comorbidities", "Comorbidities", "Comorbidities"],
            "Parameter": ["Gender", "Readmission Count", "BMI",
                         "Hematocrit", "Neutrophils", "Sodium", "Glucose", 
                         "BUN", "Creatinine", "Hemoglobin",
                         "Heart Rate", "Respiration",
                         "Dialysis/Renal", "Asthma", "Iron Deficiency", "Pneumonia",
                         "Substance Dependence", "Major Psychiatric", "Depression",
                         "Other Psychiatric", "Fibrosis", "Malnutrition"],
            "Value": [gender, rcount_display, bmi_display,
                     f"{hematocrit:.1f} %", f"{neutrophils:.1f} ×10³/µL", 
                     f"{sodium:.1f} mEq/L", f"{glucose:.1f} mg/dL",
                     f"{bloodureanitro:.1f} mg/dL", f"{creatinine:.2f} mg/dL",
                     f"{hemo:.1f} g/dL",
                     f"{pulse} bpm", f"{respiration:.1f} /min",
                     "✓" if dialysis else "✗", "✓" if asthma else "✗",
                     "✓" if irondef else "✗", "✓" if pneum else "✗",
                     "✓" if substance else "✗", "✓" if psychmajor else "✗",
                     "✓" if depress else "✗", "✓" if psychother else "✗",
                     "✓" if fibrosis else "✗", "✓" if malnutrition else "✗"]
        }
        
        df_summary = pd.DataFrame(summary_data)
        st.dataframe(df_summary, use_container_width=True, hide_index=True, height=550)
    
    with col2:
        st.markdown('<div class="section-header">🎯 Prediction Results</div>', unsafe_allow_html=True)
        
        if predict_button:
            # Prepare input data for model
            rcount_num = int(rcount.replace('5+', '5'))
            
            input_dict = {
                "rcount": rcount_num,
                "gender": gender,
                "bmi": bmi if bmi > 0 else 25.0,  # Use default if BMI invalid
                "dialysisrenalendstage": dialysis,
                "asthma": asthma,
                "irondef": irondef,
                "pneum": pneum,
                "substancedependence": substance,
                "psychologicaldisordermajor": psychmajor,
                "depress": depress,
                "psychother": psychother,
                "fibrosisandother": fibrosis,
                "malnutrition": malnutrition,
                "hemo": hemo,
                "hematocrit": hematocrit,
                "neutrophils": neutrophils,
                "sodium": sodium,
                "glucose": glucose,
                "bloodureanitro": bloodureanitro,
                "creatinine": creatinine,
                "pulse": pulse,
                "respiration": respiration
            }
            
            input_df = pd.DataFrame([input_dict])
            
            # Make prediction
            los, category, description, icon = predict_los(model, input_df, expected_features)
            
            if los is not None:
                # Display prediction card with whole number
                st.markdown(f"""
                <div class="prediction-card">
                    <div class="prediction-label">PREDICTED LENGTH OF STAY</div>
                    <div class="prediction-number">{los}</div>
                    <div style="font-size: 1.2rem; margin-top: -0.5rem;">days</div>
                    <div class="stay-category">
                        {icon} {category} {icon}
                    </div>
                    <div style="margin-top: 1rem; font-size: 0.9rem;">
                        {description}
                    </div>
                </div>
                """, unsafe_allow_html=True)
                
                # Risk factors
                active_risks = get_risk_factors(input_dict)
                if active_risks:
                    st.markdown("#### ⚠️ Active Risk Factors")
                    risk_chips = "".join(f'<span class="risk-chip">{risk}</span>' for risk in active_risks)
                    st.markdown(risk_chips, unsafe_allow_html=True)
                    
                    risk_count = len(active_risks)
                    if risk_count >= 3:
                        st.warning(f"⚠️ {risk_count} comorbidities detected. Consider enhanced care coordination.")
                
                # BMI insights (only if valid)
                if bmi > 0:
                    if bmi < 18.5:
                        st.warning("📉 Underweight BMI - Monitor nutritional status")
                    elif bmi > 30:
                        st.warning("📈 Obese BMI - Consider weight management and associated risks")
                    elif bmi > 25:
                        st.info("📊 Overweight BMI - Monitor for metabolic complications")
                
                # Clinical recommendations based on LOS
                st.markdown("#### 💡 Clinical Insights")
                if los <= 3:
                    st.info("""
                    **Recommendations:**
                    - Routine discharge planning
                    - Standard post-discharge follow-up
                    - Monitor for early readmission risk factors
                    """)
                elif los <= 7:
                    st.info("""
                    **Recommendations:**
                    - Coordinate with case management
                    - Ensure discharge criteria are met
                    - Schedule follow-up within 7 days
                    """)
                elif los <= 14:
                    st.warning("""
                    **Recommendations:**
                    - Involve multidisciplinary team
                    - Consider rehabilitation services
                    - Detailed discharge planning required
                    """)
                else:
                    st.error("""
                    **Recommendations:**
                    - Intensive case management needed
                    - Consider skilled nursing facility
                    - Complex care coordination required
                    - Regular team meetings recommended
                    """)
                
                # Visual indicator
                st.markdown("#### 📊 Stay Duration Indicator")
                progress_value = min(los / 21, 1.0)
                st.progress(progress_value, text=f"Stay duration: {los} days")
                
        else:
            st.info("👈 **Ready for prediction**\n\nFill in all patient information in the sidebar and click 'Predict Length of Stay' to see results.")
            
            # Show example or demo message
            st.markdown("""
            <div class="info-box">
                <strong>📌 How to use:</strong><br>
                1. Enter patient demographics including height/weight for BMI<br>
                2. Input lab results and vital signs (any values allowed)<br>
                3. Select comorbidities (1 = present, 0 = absent)<br>
                4. Click the predict button to see results<br>
                5. Review risk factors and clinical insights
            </div>
            """, unsafe_allow_html=True)
            
            # Display model information
            with st.expander("ℹ️ About the Model"):
                st.markdown("""
                **Model Details:**
                - **Algorithm:** CatBoost Regressor
                - **Features:** 22 clinical parameters
                - **Target:** Length of Stay (days)
                - **Training Data:** Historical hospital admissions
                
                **Key Features:**
                - Demographics (gender, BMI, readmission count)
                - Lab values (CBC, metabolic panel)
                - Vital signs (heart rate, respiration)
                - Comorbidities (10 conditions)
                
                **BMI Categories:**
                - Underweight: < 18.5
                - Normal: 18.5 - 24.9
                - Overweight: 25 - 29.9
                - Obese: ≥ 30
                
                **Output Format:**
                - Predicted length of stay is rounded to the nearest whole day
                - Categories help interpret the expected care intensity
                
                **Note:** Lab values and vital signs can accept any numeric value with no restrictions.
                """)
    
    # Footer disclaimer
    st.markdown("---")
    st.markdown("""
    <div style="background: #fff3e0; padding: 1rem; border-radius: 8px; margin-top: 1rem;">
        <small>
        ⚠️ <strong>Clinical Disclaimer:</strong> This tool is for informational and research purposes only. 
        Predictions are based on statistical models and should not replace clinical judgment. 
        Always consult with qualified healthcare professionals for medical decisions.
        </small>
    </div>
    """, unsafe_allow_html=True)

if __name__ == "__main__":
    main()

Overwriting app.py
